# Customer Segmentation with RFM and K-Means

## Objectives

This notebook will:

- create customer-level Recency, Frequency and Monetary features;
- prepare the highly skewed RFM features for machine learning;
- evaluate different numbers of clusters;
- implement K-Means customer segmentation;
- profile and name the resulting customer segments;
- recommend targeted marketing actions for each segment;
- export customer segments for the Streamlit dashboard.

## Analytical problem

The retailer needs an evidence-based method for grouping customers according to their purchasing behaviour. The resulting segments can support more relevant marketing campaigns, retention activity, and customer-value strategies.

## Methodology selection

Customer segmentation is an unsupervised machine-learning problem because the dataset does not contain existing customer-segment labels or a known target variable.

K-Means clustering was selected because:

- it is suitable for grouping numeric customer features;
- its clusters can be profiled and explained to business users;
- it is widely supported by Scikit-learn;
- it can assign each customer to one distinct segment;
- the result can be integrated into an interactive Streamlit dashboard.

RFM features were selected because they represent three important aspects of customer behaviour:

- **Recency:** days since the customer's most recent completed purchase;
- **Frequency:** number of unique completed invoices;
- **Monetary:** total completed-sales revenue generated by the customer.

Customer identifier `15287` has already been excluded from the customer-sales dataset because it appears to represent multiple unknown customers.

In [1]:
import os
from pathlib import Path

os.environ.setdefault("LOKY_MAX_CPU_COUNT", "4")

import numpy as np
import pandas as pd
import plotly.express as px
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.2f}".format)

In [2]:
project_root = Path.cwd()

if project_root.name == "jupyter_notebooks":
    project_root = project_root.parent

customer_sales_path = (
    project_root
    / "data"
    / "processed"
    / "customer_sales.parquet"
)

assert customer_sales_path.exists()

customer_sales = pd.read_parquet(customer_sales_path)

assert len(customer_sales) == 392_672
assert customer_sales["CustomerID"].nunique() == 4_337
assert not customer_sales["CustomerID"].eq(15_287).any()

print(f"Customer-sales rows: {len(customer_sales):,}")
print(
    "Reliable customer identifiers: "
    f"{customer_sales['CustomerID'].nunique():,}"
)

Customer-sales rows: 392,672
Reliable customer identifiers: 4,337


## RFM feature engineering

The analysis date is set to one day after the final transaction date. This makes a customer who purchased on the final recorded date have a recency of one day rather than zero days.

Frequency is calculated using unique invoices rather than product-line rows. This prevents invoices containing several products from being counted as several purchases.

In [3]:
analysis_date = (
    customer_sales["InvoiceDate"].max().normalize()
    + pd.Timedelta(days=1)
)

customer_rfm = (
    customer_sales.groupby(
        "CustomerID",
        as_index=False,
    )
    .agg(
        LastPurchase=("InvoiceDate", "max"),
        Frequency=("InvoiceNo", "nunique"),
        Monetary=("LineRevenue", "sum"),
    )
)

customer_rfm["Recency"] = (
    analysis_date
    - customer_rfm["LastPurchase"].dt.normalize()
).dt.days

customer_rfm = customer_rfm[
    [
        "CustomerID",
        "Recency",
        "Frequency",
        "Monetary",
        "LastPurchase",
    ]
]

print(f"Analysis date: {analysis_date.date()}")
customer_rfm.head()

Analysis date: 2011-12-10


,CustomerID,Recency,Frequency,Monetary,LastPurchase
0,12346,326,1,"77,183.60",2011-01-18 10:01:00
1,12347,3,7,"4,310.00",2011-12-07 15:52:00
2,12348,76,4,"1,797.24",2011-09-25 13:13:00
3,12349,19,1,"1,757.55",2011-11-21 09:51:00
4,12350,311,1,334.40,2011-02-02 16:01:00


In [4]:
rfm_statistics = customer_rfm[
    [
        "Recency",
        "Frequency",
        "Monetary",
    ]
].describe(
    percentiles=[
        0.25,
        0.50,
        0.75,
        0.90,
        0.95,
        0.99,
    ]
)

rfm_statistics

,Recency,Frequency,Monetary
count,"4,337.00","4,337.00","4,337.00"
mean,93.08,4.27,"2,049.05"
std,100.02,7.70,"8,986.23"
min,1.00,1.00,3.75
25%,18.00,1.00,306.46
50%,51.00,2.00,668.58
75%,143.00,5.00,"1,660.88"
90%,263.40,9.00,"3,640.90"
95%,312.00,13.00,"5,792.76"
99%,369.64,30.00,"19,780.71"


### RFM distribution interpretation

Frequency and Monetary contain substantial positive skew. Most customers made relatively few purchases and generated modest revenue, while a small number of customers purchased very frequently or generated exceptionally high revenue.

Recency is also unevenly distributed. Because K-Means uses distances between observations, highly skewed variables and extreme values could dominate the clustering result.

A `log1p` transformation will reduce skew while retaining every customer. Standardisation will then place the three transformed features on comparable scales.

In [5]:
rfm_chart_data = customer_rfm.melt(
    id_vars="CustomerID",
    value_vars=[
        "Recency",
        "Frequency",
        "Monetary",
    ],
    var_name="RFMFeature",
    value_name="FeatureValue",
)

fig = px.histogram(
    rfm_chart_data,
    x="FeatureValue",
    facet_col="RFMFeature",
    facet_col_wrap=3,
    nbins=50,
    title="Customer RFM Feature Distributions",
    labels={
        "FeatureValue": "Feature value",
        "RFMFeature": "RFM feature",
    },
)

fig.update_layout(
    showlegend=False,
)

fig.for_each_annotation(
    lambda annotation: annotation.update(
        text=annotation.text.split("=")[-1]
    )
)

fig.show()

In [6]:
rfm_feature_names = [
    "Recency",
    "Frequency",
    "Monetary",
]

rfm_model_features = customer_rfm[
    rfm_feature_names
].copy()

rfm_log_features = np.log1p(
    rfm_model_features
)

feature_scaler = StandardScaler()

scaled_rfm_features = feature_scaler.fit_transform(
    rfm_log_features
)

scaled_rfm_summary = pd.DataFrame(
    scaled_rfm_features,
    columns=rfm_feature_names,
).agg(
    [
        "mean",
        "std",
        "min",
        "max",
    ]
)

scaled_rfm_summary

,Recency,Frequency,Monetary
mean,-0.00,-0.00,-0.00
std,1.00,1.00,1.00
min,-2.42,-0.96,-4.00
max,1.58,5.86,4.73


## Selecting the number of clusters

Models containing between two and eight clusters will be compared using:

- **Inertia:** the total squared distance between observations and their assigned cluster centres. Lower values indicate more compact clusters, but inertia always decreases when more clusters are added.
- **Silhouette score:** compares cohesion within clusters with separation between clusters. Values closer to one indicate better-defined clusters.

The final choice should consider both statistical performance and whether the segments are sufficiently detailed and actionable for the business.

In [7]:
cluster_evaluation_records = []

for number_of_clusters in range(2, 9):
    candidate_model = KMeans(
        n_clusters=number_of_clusters,
        random_state=42,
        n_init=20,
    )

    candidate_labels = candidate_model.fit_predict(
        scaled_rfm_features
    )

    cluster_evaluation_records.append(
        {
            "NumberOfClusters": number_of_clusters,
            "Inertia": candidate_model.inertia_,
            "SilhouetteScore": silhouette_score(
                scaled_rfm_features,
                candidate_labels,
            ),
        }
    )

cluster_evaluation = pd.DataFrame(
    cluster_evaluation_records
)

cluster_evaluation

,NumberOfClusters,Inertia,SilhouetteScore
0,2,"6,474.11",0.43
1,3,"4,856.77",0.34
2,4,"3,924.57",0.33
3,5,"3,268.21",0.32
4,6,"2,839.52",0.32
5,7,"2,532.19",0.31
6,8,"2,324.03",0.30


In [8]:
fig = px.line(
    cluster_evaluation,
    x="NumberOfClusters",
    y="Inertia",
    markers=True,
    title="K-Means Elbow Analysis",
    labels={
        "NumberOfClusters": "Number of clusters",
        "Inertia": "Model inertia",
    },
)

fig.update_traces(
    line={"width": 3},
    marker={"size": 9},
)

fig.update_xaxes(
    dtick=1,
)

fig.show()

In [9]:
fig = px.line(
    cluster_evaluation,
    x="NumberOfClusters",
    y="SilhouetteScore",
    markers=True,
    title="K-Means Silhouette Scores",
    labels={
        "NumberOfClusters": "Number of clusters",
        "SilhouetteScore": "Silhouette score",
    },
)

fig.update_traces(
    line={"width": 3},
    marker={"size": 9},
)

fig.update_xaxes(
    dtick=1,
)

fig.show()

### Cluster-count decision

The two-cluster solution produces the highest silhouette score, indicating the strongest statistical separation. However, two clusters would provide only a broad division between higher-value and lower-value customers and would offer limited detail for targeted marketing.

The elbow chart shows that improvements in inertia become progressively smaller after approximately four clusters. A four-cluster solution retains a positive silhouette score of approximately 0.333 while allowing the retailer to distinguish several commercially meaningful patterns.

Therefore, **four clusters** will be selected as a balance between statistical separation, simplicity, and business usefulness. The lower silhouette score compared with the two-cluster model will be recorded as a limitation rather than hidden.

In [10]:
assert len(customer_rfm) == 4_337
assert customer_rfm["CustomerID"].is_unique

assert customer_rfm["Recency"].ge(1).all()
assert customer_rfm["Frequency"].ge(1).all()
assert customer_rfm["Monetary"].gt(0).all()

assert scaled_rfm_features.shape == (4_337, 3)
assert np.isfinite(scaled_rfm_features).all()

assert cluster_evaluation["NumberOfClusters"].tolist() == list(
    range(2, 9)
)

assert cluster_evaluation["SilhouetteScore"].between(
    -1,
    1,
).all()

print("RFM preparation and model evaluation validated successfully.")

RFM preparation and model evaluation validated successfully.


## Final K-Means customer segmentation

The selected four-cluster model will now be trained using all 4,337 reliable customers.

The model uses the log-transformed and standardised RFM features created in the previous section. A fixed random state and explicit `n_init` value make the result reproducible.

In [11]:
selected_cluster_count = 4

final_kmeans_model = KMeans(
    n_clusters=selected_cluster_count,
    random_state=42,
    n_init=20,
)

customer_rfm["Cluster"] = final_kmeans_model.fit_predict(
    scaled_rfm_features
)

final_silhouette_score = silhouette_score(
    scaled_rfm_features,
    customer_rfm["Cluster"],
)

model_performance = pd.Series(
    {
        "Selected clusters": selected_cluster_count,
        "Model inertia": final_kmeans_model.inertia_,
        "Silhouette score": final_silhouette_score,
    }
)

model_performance

Selected clusters       4.00
Model inertia       3,924.57
Silhouette score        0.33
dtype: float64

In [12]:
initial_cluster_profile = (
    customer_rfm.groupby(
        "Cluster",
        as_index=False,
    )
    .agg(
        Customers=("CustomerID", "size"),
        MeanRecency=("Recency", "mean"),
        MedianRecency=("Recency", "median"),
        MeanFrequency=("Frequency", "mean"),
        MedianFrequency=("Frequency", "median"),
        MeanMonetary=("Monetary", "mean"),
        MedianMonetary=("Monetary", "median"),
        TotalRevenue=("Monetary", "sum"),
    )
)

initial_cluster_profile

,Cluster,Customers,MeanRecency,MedianRecency,MeanFrequency,MedianFrequency,MeanMonetary,MedianMonetary,TotalRevenue
0,0,1175,64.76,52.00,4.20,4.00,"1,832.39","1,366.86","2,153,056.28"
1,1,1558,191.78,186.00,1.36,1.00,362.14,304.25,"564,221.00"
2,2,893,22.54,20.00,1.93,2.00,486.93,414.20,"434,828.04"
3,3,711,12.17,9.00,13.72,10.00,"8,065.57","3,701.44","5,734,616.88"


### Segment naming

Cluster names are assigned after examining their average and median RFM values.

- **High-Value Loyal:** very recent, frequent and high-value customers.
- **Established Regulars:** moderately recent customers with repeated purchases and meaningful spending.
- **Recent Low-Frequency:** recent customers with relatively few purchases and lower spending.
- **Inactive Low-Value:** customers who have not purchased recently and have low frequency and monetary value.

These names describe observed behaviour. They do not describe customer demographics, preferences, or motivations.

In [13]:
segment_mapping = {
    0: "Established Regulars",
    1: "Inactive Low-Value",
    2: "Recent Low-Frequency",
    3: "High-Value Loyal",
}

customer_rfm["Segment"] = customer_rfm[
    "Cluster"
].map(segment_mapping)

assert customer_rfm["Segment"].notna().all()

customer_rfm.head()

,CustomerID,Recency,Frequency,Monetary,LastPurchase,Cluster,Segment
0,12346,326,1,"77,183.60",2011-01-18 10:01:00,0,Established Regulars
1,12347,3,7,"4,310.00",2011-12-07 15:52:00,3,High-Value Loyal
2,12348,76,4,"1,797.24",2011-09-25 13:13:00,0,Established Regulars
3,12349,19,1,"1,757.55",2011-11-21 09:51:00,2,Recent Low-Frequency
4,12350,311,1,334.40,2011-02-02 16:01:00,1,Inactive Low-Value


In [14]:
segment_profile = (
    customer_rfm.groupby(
        [
            "Cluster",
            "Segment",
        ],
        as_index=False,
    )
    .agg(
        Customers=("CustomerID", "size"),
        MeanRecency=("Recency", "mean"),
        MedianRecency=("Recency", "median"),
        MeanFrequency=("Frequency", "mean"),
        MedianFrequency=("Frequency", "median"),
        MeanMonetary=("Monetary", "mean"),
        MedianMonetary=("Monetary", "median"),
        TotalRevenue=("Monetary", "sum"),
    )
)

segment_profile["CustomerSharePercentage"] = (
    segment_profile["Customers"]
    / len(customer_rfm)
    * 100
)

segment_profile["RevenueSharePercentage"] = (
    segment_profile["TotalRevenue"]
    / customer_rfm["Monetary"].sum()
    * 100
)

segment_profile.sort_values(
    "MeanMonetary",
    ascending=False,
)

,Cluster,Segment,Customers,MeanRecency,MedianRecency,MeanFrequency,MedianFrequency,MeanMonetary,MedianMonetary,TotalRevenue,CustomerSharePercentage,RevenueSharePercentage
3,3,High-Value Loyal,711,12.17,9.00,13.72,10.00,"8,065.57","3,701.44","5,734,616.88",16.39,64.53
0,0,Established Regulars,1175,64.76,52.00,4.20,4.00,"1,832.39","1,366.86","2,153,056.28",27.09,24.23
2,2,Recent Low-Frequency,893,22.54,20.00,1.93,2.00,486.93,414.20,"434,828.04",20.59,4.89
1,1,Inactive Low-Value,1558,191.78,186.00,1.36,1.00,362.14,304.25,"564,221.00",35.92,6.35


In [15]:
segment_share_chart = segment_profile.melt(
    id_vars="Segment",
    value_vars=[
        "CustomerSharePercentage",
        "RevenueSharePercentage",
    ],
    var_name="Measure",
    value_name="Percentage",
)

segment_share_chart["Measure"] = (
    segment_share_chart["Measure"].replace(
        {
            "CustomerSharePercentage": "Customer share",
            "RevenueSharePercentage": "Revenue share",
        }
    )
)

segment_order = [
    "High-Value Loyal",
    "Established Regulars",
    "Recent Low-Frequency",
    "Inactive Low-Value",
]

fig = px.bar(
    segment_share_chart,
    x="Segment",
    y="Percentage",
    color="Measure",
    barmode="group",
    category_orders={
        "Segment": segment_order,
    },
    title="Customer Share and Revenue Share by Segment",
    labels={
        "Segment": "Customer segment",
        "Percentage": "Share (%)",
        "Measure": "Measure",
    },
    text_auto=".1f",
)

fig.update_yaxes(
    ticksuffix="%",
)

fig.update_layout(
    xaxis_title=None,
    legend_title_text="",
)

fig.show()

In [16]:
segment_colours = {
    "High-Value Loyal": "#2E7D32",
    "Established Regulars": "#1976D2",
    "Recent Low-Frequency": "#F9A825",
    "Inactive Low-Value": "#C62828",
}

fig = px.scatter_3d(
    customer_rfm,
    x="Recency",
    y="Frequency",
    z="Monetary",
    color="Segment",
    color_discrete_map=segment_colours,
    category_orders={
        "Segment": segment_order,
    },
    log_x=True,
    log_y=True,
    log_z=True,
    opacity=0.65,
    title="Customer Segments Across RFM Dimensions",
    labels={
        "Recency": "Recency in days (logarithmic)",
        "Frequency": "Completed invoices (logarithmic)",
        "Monetary": "Customer revenue (£, logarithmic)",
        "Segment": "Customer segment",
    },
    hover_data={
        "CustomerID": True,
        "LastPurchase": True,
        "Cluster": False,
    },
)

fig.update_layout(
    legend_title_text="Customer segment",
)

fig.show()

In [17]:
cluster_centres = pd.DataFrame(
    final_kmeans_model.cluster_centers_,
    columns=rfm_feature_names,
)

cluster_centres["Cluster"] = range(
    selected_cluster_count
)

cluster_centres["Segment"] = cluster_centres[
    "Cluster"
].map(segment_mapping)

cluster_centre_chart = (
    cluster_centres.set_index("Segment")[
        rfm_feature_names
    ]
    .reindex(segment_order)
)

fig = px.imshow(
    cluster_centre_chart,
    text_auto=".2f",
    aspect="auto",
    color_continuous_scale="RdBu",
    color_continuous_midpoint=0,
    title="Standardised RFM Cluster Profiles",
    labels={
        "x": "RFM feature",
        "y": "Customer segment",
        "color": "Standardised centre",
    },
)

fig.show()

## Segment interpretation and marketing recommendations

### High-Value Loyal

This segment contains approximately 16.39% of customers but generates about 64.53% of customer-attributed revenue.

Recommended actions:

- provide early access to new or limited products;
- use personalised product recommendations;
- offer loyalty benefits that encourage retention;
- monitor declining purchase frequency as an early churn signal;
- avoid unnecessary blanket discounts that reduce margin.

### Established Regulars

These customers purchase repeatedly and generate approximately 24.23% of customer-attributed revenue.

Recommended actions:

- use complementary-product and cross-selling recommendations;
- introduce loyalty milestones;
- encourage movement toward the High-Value Loyal segment;
- promote relevant seasonal collections.

### Recent Low-Frequency

These customers purchased recently but have completed relatively few orders.

Recommended actions:

- use second-purchase campaigns;
- recommend products related to their first or recent purchase;
- provide time-limited incentives where commercially appropriate;
- introduce the retailer's loyalty programme.

### Inactive Low-Value

This is the largest segment by customer count but contributes a relatively small share of revenue.

Recommended actions:

- use low-cost automated re-engagement campaigns;
- test win-back messages on a limited sample;
- avoid expensive incentives without evidence of likely return;
- reduce marketing frequency when customers repeatedly do not respond.

The segmentation provides targeting guidance, but campaign effectiveness should be evaluated through controlled tests using conversion rate, incremental revenue, and marketing cost.

## Model effectiveness and limitations

The four-cluster model produces interpretable and commercially actionable segments. The High-Value Loyal segment is particularly useful because it
identifies a relatively small customer group responsible for most customer-attributed revenue.

The silhouette score of approximately 0.333 is positive but moderate. This indicates that the clusters have useful structure but are not completely separated. A two-cluster model achieved a higher silhouette score, so the four-cluster selection prioritises business detail over maximum statistical separation.

Additional limitations include:

- K-Means favours compact, approximately spherical clusters;
- results are sensitive to feature selection and scaling;
- segment membership can change when new transactions are added;
- the dataset covers a limited historical period;
- repeat invoices from the same customer are summarised but customer behaviour may change over time;
- the dataset contains no demographics, campaign responses, profit margins, or product categories;
- segment associations do not establish that a marketing action will cause increased sales.

Alternative methods could include hierarchical clustering, DBSCAN, Gaussian mixture models, or rule-based RFM scoring. Future work should compare cluster stability over time and measure campaign outcomes for each segment.

## Export for the Streamlit dashboard

The customer-level segmentation will be stored in the processed-data directory. The original raw CSV remains unchanged.

In [18]:
customer_segments_path = (
    project_root
    / "data"
    / "processed"
    / "customer_segments.parquet"
)

customer_segment_output = customer_rfm[
    [
        "CustomerID",
        "Recency",
        "Frequency",
        "Monetary",
        "LastPurchase",
        "Cluster",
        "Segment",
    ]
].copy()

customer_segment_output.to_parquet(
    customer_segments_path,
    index=False,
)

exported_customer_segments = pd.read_parquet(
    customer_segments_path
)

assert len(exported_customer_segments) == len(
    customer_segment_output
)

assert exported_customer_segments[
    "CustomerID"
].is_unique

assert exported_customer_segments[
    "Segment"
].notna().all()

assert np.isclose(
    exported_customer_segments["Monetary"].sum(),
    customer_segment_output["Monetary"].sum(),
)

print(
    "Customer segments exported successfully: "
    f"{customer_segments_path}"
)

Customer segments exported successfully: /Users/ewa/Documents/vscode-projects/code-institute-2026/CI-Project2-Online Retail Transaction Analysis/CI-DA-Project-2-Online-Retail-Transaction-Analysis/data/processed/customer_segments.parquet


In [19]:
assert customer_rfm["Cluster"].nunique() == 4
assert customer_rfm["Segment"].nunique() == 4

assert segment_profile["Customers"].sum() == 4_337

assert np.isclose(
    segment_profile["CustomerSharePercentage"].sum(),
    100,
)

assert np.isclose(
    segment_profile["RevenueSharePercentage"].sum(),
    100,
)

assert final_silhouette_score > 0

assert (
    segment_profile.loc[
        segment_profile["Segment"].eq("High-Value Loyal"),
        "MeanMonetary",
    ].iloc[0]
    == segment_profile["MeanMonetary"].max()
)

print("Final customer segmentation validated successfully.")

Final customer segmentation validated successfully.
